In [ ]:
# Worked Example: DatetimeIndex, Log Inflation Rates & Monthly Resampling
import pandas as pd
import numpy as np

# 1. Create a sample daily economic price DataFrame
sample_dates = pd.date_range(start='2025-01-01', periods=5, freq='B')
sample_prices = pd.DataFrame({
    'Date': sample_dates.strftime('%Y-%m-%d'),
    'Close': [150.0, 153.0, 151.5, 155.0, 158.0]
})

# 2. Convert Date to DatetimeIndex
sample_prices['Date'] = pd.to_datetime(sample_prices['Date'])
sample_prices = sample_prices.set_index('Date')

# 3. Compute Simple and Log Inflation/Return Rates
sample_prices['Simple_Return'] = sample_prices['Close'].pct_change()
sample_prices['Log_Return'] = np.log(sample_prices['Close'] / sample_prices['Close'].shift(1))

print("Daily Macroeconomic Series with Inflation Rates:")
print(sample_prices)

# 4. Resample to Monthly Frequency (taking the last price observation of each month)
monthly_prices = sample_prices[['Close']].resample('ME').last()
print("\nMonthly Resampled Macroeconomic Series:")
print(monthly_prices)


Daily Macroeconomic Series with Inflation Rates:
            Close  Simple_Return  Log_Return
Date                                        
2025-01-01  150.0            NaN         NaN
2025-01-02  153.0       0.020000    0.019803
2025-01-03  151.5      -0.009804   -0.009852
2025-01-06  155.0       0.023102    0.022839
2025-01-07  158.0       0.019355    0.019170

Monthly Resampled Macroeconomic Series:
            Close
Date             
2025-01-31  158.0


In [ ]:
import pandas as pd
import numpy as np

def analyze_macro_series_and_resample(
    df: pd.DataFrame,
    start_date: str = '2025-01-01',
    end_date: str = '2025-05-31'
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Converts Date column to DatetimeIndex, slices date window, computes simple & log returns,
    and resamples daily Close prices to monthly end ('ME') frequency.

    Parameters:
        df (pd.DataFrame): Daily DataFrame containing 'Date' and 'Close' columns.
        start_date (str): Beginning of analysis period (inclusive).
        end_date (str): End of analysis period (inclusive).

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: (sliced_daily_df, monthly_df)
    """
    # Log returns are calculated because they are time-additive,
    # meaning the sum of daily log returns equals the total log return
    # over the entire investment period.
    try:
        # Make a copy to avoid modifying the original DataFrame passed to the function
        processed_df = df.copy()

        # Convert Date to DatetimeIndex and set as index for the input DataFrame
        processed_df['Date'] = pd.to_datetime(processed_df['Date'])
        processed_df = processed_df.set_index('Date')

        # Slice the DataFrame based on the date window
        sliced_df = processed_df.loc[start_date:end_date].copy()

        # Compute Simple and Log Returns
        sliced_df['Simple_Return'] = sliced_df['Close'].pct_change()
        sliced_df['Log_Return'] = np.log(sliced_df['Close'] / sliced_df['Close'].shift(1))

        # Resample to Monthly Frequency (taking the last price observation of each month)
        monthly_df = sliced_df[['Close']].resample('ME').last()
        monthly_df['Log_Return_Monthly'] = np.log(monthly_df['Close'] / monthly_df['Close'].shift(1))

        return sliced_df, monthly_df
    except Exception as e:
        print(f"Error in analyze_macro_series_and_resample: {e}")
        return pd.DataFrame(), pd.DataFrame()

# --- Example Usage --- #
# Create a synthetic daily stock DataFrame
np.random.seed(42)
synth_dates = pd.date_range('2025-01-01', periods=150, freq='B')
synth_returns = np.random.normal(0.0005, 0.015, size=150)
synth_prices = 100 * np.exp(np.cumsum(synth_returns))
raw_stock_df = pd.DataFrame({'Date': synth_dates.strftime('%Y-%m-%d'), 'Close': synth_prices})

try:
    daily_res, monthly_res = analyze_macro_series_and_resample(raw_stock_df.copy(), '2025-01-01', '2025-05-31')

    print("--- 1. Sliced Daily Series with Returns (First 5 Rows) ---")
    display(daily_res.head())

    print("\n--- 2. Monthly Resampled Series ('ME' Frequency) ---")
    display(monthly_res)

    print("\n--- 3. Time Additivity Verification ---")
    # For time additivity of log returns, the sum of daily log returns should approximate
    # the log of the ratio of the last price to the first price over the same period.
    sum_log_ret = daily_res['Log_Return'].sum()
    total_log_change = np.log(daily_res['Close'].iloc[-1] / daily_res['Close'].iloc[0])
    print(f"Sum of daily log returns: {sum_log_ret:.6f}")
    print(f"Total period log change:  {total_log_change:.6f}")
    print(f"Difference:               {abs(sum_log_ret - total_log_change):.2e}")
except Exception as e:
    print(f"Inspection notice: {e}")


--- 1. Sliced Daily Series with Returns (First 5 Rows) ---


,Close,Simple_Return,Log_Return
Date,,,
2025-01-01,100.798240,NaN,NaN
2025-01-02,100.639712,-0.001573,-0.001574
2025-01-03,101.673049,0.010268,0.010215
2025-01-06,104.074575,0.023620,0.023345
2025-01-07,103.761543,-0.003008,-0.003012



--- 2. Monthly Resampled Series ('ME' Frequency) ---


,Close,Log_Return_Monthly
Date,,
2025-01-31,97.992567,NaN
2025-02-28,90.685809,-0.077491
2025-03-31,85.924616,-0.053931
2025-04-30,89.690442,0.042894
2025-05-31,89.410509,-0.003126



--- 3. Time Additivity Verification ---
Sum of daily log returns: -0.119883
Total period log change:  -0.119883
Difference:               2.91e-16


In [ ]:
# Test cell for Exercise 1
# Run this cell to verify your implementation!

try:
    # Generate synthetic daily price series
    np.random.seed(42)
    synth_dates = pd.date_range('2025-01-01', periods=150, freq='B')
    synth_returns = np.random.normal(0.0005, 0.015, size=150)
    synth_prices = 100 * np.exp(np.cumsum(synth_returns))
    raw_stock_df = pd.DataFrame({
        'Date': synth_dates.strftime('%Y-%m-%d'),
        'Close': synth_prices
    })

    daily_res, monthly_res = analyze_macro_series_and_resample(raw_stock_df, '2025-01-01', '2025-05-31')

    # 1. Verify DatetimeIndex
    assert isinstance(daily_res.index, pd.DatetimeIndex), "Index of daily DataFrame must be a DatetimeIndex."
    assert daily_res.index.min() == pd.Timestamp('2025-01-01'), "Date slicing lower bound failed."
    assert daily_res.index.max() <= pd.Timestamp('2025-05-31'), "Date slicing upper bound failed."

    # 2. Verify Return Columns
    assert 'Simple_Return' in daily_res.columns, "Simple_Return column is missing."
    assert 'Log_Return' in daily_res.columns, "Log_Return column is missing."
    assert pd.isna(daily_res['Simple_Return'].iloc[0]), "First daily simple return should be NaN."
    assert pd.isna(daily_res['Log_Return'].iloc[0]), "First daily log return should be NaN."

    # 3. Verify Log Return Time Additivity
    sum_log_ret = daily_res['Log_Return'].sum()
    total_log_change = np.log(daily_res['Close'].iloc[-1] / daily_res['Close'].iloc[0])
    assert abs(sum_log_ret - total_log_change) < 1e-6, "Log returns are not time-additive!"

    # 4. Verify Monthly Resampling
    assert monthly_res.shape[0] == 5, f"Expected 5 monthly periods (Jan-May), got {monthly_res.shape[0]}"
    assert monthly_res.index.freqstr in ['ME', 'M'], "Monthly resample frequency should be 'ME'."

    print("🎉 All Part 1 tests passed!")
except AssertionError as e:
    print(f"❌ Verification failed: {e}")
except Exception as e:
    print(f"❌ Unexpected execution error: {e}")


🎉 All Part 1 tests passed!


In [ ]:
# Worked Example: Rolling Statistics & Maximum Drawdown
# Using a sample economic index DataFrame
risk_df = pd.DataFrame({
    'Close': [100.0, 105.0, 102.0, 98.0, 95.0, 103.0, 108.0]
})

# 1. Compute 3-day Rolling Mean (SMA)
risk_df['SMA_3D'] = risk_df['Close'].rolling(window=3).mean()

# 2. Compute Cumulative Peak Price
risk_df['Peak'] = risk_df['Close'].cummax()

# 3. Compute Percentage Drawdown from Peak
risk_df['Drawdown'] = (risk_df['Close'] - risk_df['Peak']) / risk_df['Peak']

# 4. Extract Maximum Drawdown (MDD)
mdd = risk_df['Drawdown'].min()

print("DataFrame with Rolling SMA & Drawdown:")
print(risk_df)
print(f"\nMaximum Peak-to-Trough Drawdown: {mdd * 100:.2f}%")


DataFrame with Rolling SMA & Drawdown:
   Close      SMA_3D   Peak  Drawdown
0  100.0         NaN  100.0  0.000000
1  105.0         NaN  105.0  0.000000
2  102.0  102.333333  105.0 -0.028571
3   98.0  101.666667  105.0 -0.066667
4   95.0   98.333333  105.0 -0.095238
5  103.0   98.666667  105.0 -0.019048
6  108.0  102.000000  108.0  0.000000

Maximum Peak-to-Trough Drawdown: -9.52%


In [ ]:
def compute_risk_metrics(df: pd.DataFrame) -> tuple[pd.DataFrame, float]:
    """
    Computes technical and risk metrics for a financial time series.

    Returns:
        Updated DataFrame and maximum drawdown (scalar float).
    """

    # Create a copy to preserve the original DataFrame
    df = df.copy()

    # Calculate 20-day Simple Moving Average (SMA)
    df["SMA_20"] = df["Close"].rolling(window=20).mean()

    # Calculate 21-day rolling volatility from log returns
    # Annualize volatility using sqrt(252) trading days
    df["Rolling_Volatility"] = (
        df["Log_Return"]
        .rolling(window=21)
        .std()
        * np.sqrt(252)
    )

    # Calculate the running peak price observed up to each date
    df["Peak"] = df["Close"].cummax()

    # Compute percentage drawdown from the historical peak
    # Drawdown = (Current Price - Peak Price) / Peak Price
    df["Drawdown"] = (
        (df["Close"] - df["Peak"]) / df["Peak"]
    )

    # Find the worst drawdown in the sample period
    # .min() returns a scalar value
    max_drawdown = float(df["Drawdown"].min())

    # Return the updated DataFrame and scalar maximum drawdown
    return df, max_drawdown

In [ ]:
import pandas as pd
import numpy as np

def compute_rolling_macro_risk_and_drawdown(
    df: pd.DataFrame,
    window_sma: int = 20,
    window_vol: int = 21
) -> tuple[pd.DataFrame, float]:
    """
    Computes SMA_20, Rolling_Vol_21D, Peak, Drawdown, and scalar Maximum Drawdown.

    Parameters:
        df (pd.DataFrame): Daily DataFrame containing 'Close' and 'Log_Return' columns.
        window_sma (int): Rolling window for simple moving average.
        window_vol (int): Rolling window for annualized volatility.

    Returns:
        tuple[pd.DataFrame, float]: (df_with_metrics, max_drawdown)
    """
    # ==============================================================================
    # SMA_20 is used to identify the underlying trend by smoothing daily price movements.
    # Rolling_Vol_21D measures the variability of log returns over a 21-day window.
    # The volatility is annualized using sqrt(252), assuming 252 trading days per year.
    # Peak represents the highest closing price observed up to each point in time.
    # Drawdown measures the percentage decline from the historical peak.
    # Maximum Drawdown is the largest observed loss from a peak to a subsequent trough
    # and is a common downside risk measure in finance.
    # ==============================================================================

    # Create a copy to preserve the original DataFrame
    df = df.copy()

    # Calculate Simple Moving Average (SMA)
    df[f"SMA_{window_sma}"] = df["Close"].rolling(window=window_sma).mean()

    # Calculate rolling volatility from log returns
    # Annualize volatility using sqrt(252) trading days
    df["Rolling_Vol_" + str(window_vol) + "D"] = (
        df["Log_Return"]
        .rolling(window=window_vol)
        .std()
        * np.sqrt(252)
    )

    # Calculate the running peak price observed up to each date
    df["Peak"] = df["Close"].cummax()

    # Compute percentage drawdown from the historical peak
    # Drawdown = (Current Price - Peak Price) / Peak Price
    df["Drawdown"] = (
        (df["Close"] - df["Peak"]) / df["Peak"]
    )

    # Find the worst drawdown in the sample period
    # .min() returns a scalar value
    max_drawdown = float(df["Drawdown"].min())

    # Return the updated DataFrame and scalar maximum drawdown
    return df, max_drawdown


In [ ]:
try:
    # Generate synthetic daily price series
    np.random.seed(42)
    synth_dates = pd.date_range('2025-01-01', periods=150, freq='B')
    synth_returns = np.random.normal(0.0005, 0.015, size=150)
    synth_prices = 100 * np.exp(np.cumsum(synth_returns))
    raw_stock_df = pd.DataFrame({'Date': synth_dates.strftime('%Y-%m-%d'), 'Close': synth_prices})

    daily_df, _ = analyze_macro_series_and_resample(raw_stock_df.copy(), '2025-01-01', '2025-05-31')
    risk_df, max_mdd = compute_rolling_macro_risk_and_drawdown(daily_df)

    print(f"Calculated Historical Maximum Drawdown: {max_mdd * 100:.2f}%")
    print("\n--- Daily Series with Rolling Risk Indicators (Rows 18-25) ---")
    display(risk_df[['Close', 'SMA_20', 'Rolling_Vol_21D', 'Peak', 'Drawdown']].iloc[18:25])
except Exception as e:
    print(f"Inspection notice: {e}")

Calculated Historical Maximum Drawdown: -20.22%

--- Daily Series with Rolling Risk Indicators (Rows 18-25) ---


,Close,SMA_20,Rolling_Vol_21D,Peak,Drawdown
Date,,,,,
2025-01-27,97.950786,NaN,NaN,107.488015,-0.088728
2025-01-28,95.945528,102.888556,NaN,107.488015,-0.107384
2025-01-29,98.127274,102.755008,NaN,107.488015,-0.087086
2025-01-30,97.844424,102.615244,0.236373,107.488015,-0.089718
2025-01-31,97.992567,102.431219,0.236588,107.488015,-0.088340
2025-02-03,95.968538,102.025918,0.242020,107.488015,-0.107170
2025-02-04,95.235680,101.599624,0.222983,107.488015,-0.113988


In [ ]:
# Test cell for Exercise 2
# Run this cell to verify your implementation!

try:
    # Use output from Exercise 1
    np.random.seed(42)
    synth_dates = pd.date_range('2025-01-01', periods=150, freq='B')
    synth_returns = np.random.normal(0.0005, 0.015, size=150)
    synth_prices = 100 * np.exp(np.cumsum(synth_returns))
    raw_stock_df = pd.DataFrame({
        'Date': synth_dates.strftime('%Y-%m-%d'),
        'Close': synth_prices
    })

    daily_df, _ = analyze_macro_series_and_resample(raw_stock_df, '2025-01-01', '2025-05-31')
    risk_df, max_mdd = compute_rolling_macro_risk_and_drawdown(daily_df)

    # 1. Verify metric columns exist
    for col in ['SMA_20', 'Rolling_Vol_21D', 'Peak', 'Drawdown']:
        assert col in risk_df.columns, f"Column '{col}' is missing from output DataFrame."

    # 2. Verify SMA calculation
    expected_sma_20 = daily_df['Close'].iloc[0:20].mean()
    assert abs(risk_df['SMA_20'].iloc[19] - expected_sma_20) < 1e-5, "SMA_20 calculation is incorrect."

    # 3. Verify Rolling Volatility math
    expected_vol = daily_df['Log_Return'].iloc[1:22].std() * np.sqrt(252)
    assert abs(risk_df['Rolling_Vol_21D'].iloc[21] - expected_vol) < 1e-5, "Rolling_Vol_21D math is incorrect."

    # 4. Verify Drawdown and Maximum Drawdown scalar
    assert risk_df['Drawdown'].max() <= 0.0 + 1e-9, "Drawdown values cannot exceed 0."
    assert abs(max_mdd - risk_df['Drawdown'].min()) < 1e-9, "Maximum Drawdown scalar matches minimum drawdown."
    assert max_mdd < 0.0, "Max drawdown should be a negative percentage."

    print(f"🎉 All Part 2 tests passed! Max Drawdown calculated: {max_mdd * 100:.2f}%")
except AssertionError as e:
    print(f"❌ Verification failed: {e}")
except Exception as e:
    print(f"❌ Unexpected execution error: {e}")


🎉 All Part 2 tests passed! Max Drawdown calculated: -20.22%


In [ ]:
# MY COMMENTS
# Check whether all required columns are created
# Verify 20-day SMA calculation
# Verify annualized 21-day rolling volatility
# Check that drawdown values are non-positive
# Confirm maximum drawdown matches minimum drawdown
# Ensure maximum drawdown represents a loss

# This function converts the Date column into a datetime index and selects the desired date range.
# It then calculates simple and log returns from the
# Close prices and resamples the daily data into monthly end-of-period observations.


In [ ]:
# Worked Example: Vectorized Arithmetic & Custom Mapping with .apply()
sample_panel = pd.DataFrame({
    'Firm': ['Alpha Inc', 'Beta Corp', 'Gamma LLC'],
    'Sales': [1200.0, 4500.0, 300.0],
    'Assets': [2000.0, 3000.0, 500.0],
    'AltmanZ': [3.4, 2.1, 1.2]
})

# Vectorized division for Asset Turnover
sample_panel['Asset_Turnover'] = sample_panel['Sales'] / sample_panel['Assets']

# Custom classification function for Z-Score credit status
def get_credit_zone(z: float) -> str:
    if pd.isna(z):
        return "Unknown"
    elif z > 2.99:
        return "Safe"
    elif z >= 1.81:
        return "Grey Zone"
    else:
        return "Distressed"

sample_panel['Z_Status'] = sample_panel['AltmanZ'].apply(get_credit_zone)

print("Scored Sample Corporate Panel:")
print(sample_panel)


Scored Sample Corporate Panel:
        Firm   Sales  Assets  AltmanZ  Asset_Turnover    Z_Status
0  Alpha Inc  1200.0  2000.0      3.4             0.6        Safe
1  Beta Corp  4500.0  3000.0      2.1             1.5   Grey Zone
2  Gamma LLC   300.0   500.0      1.2             0.6  Distressed


In [ ]:
def get_credit_zone(z: float) -> str:
    if pd.isna(z):
        return "Unknown"
    elif z > 2.99:
        return "Safe"
    elif z >= 1.81:
        return "Grey Zone"
    else:
        return "Distressed"

In [ ]:
import pandas as pd
import numpy as np

def clean_and_score_corporate_panel(filepath: str = 'FINAL_FIGURES_PANEL.csv') -> pd.DataFrame:
    """
    Loads corporate panel CSV, cleans missing observations, filters active firm universe,
    computes vectorized financial ratios, classifies Altman Z-Score credit distress status,
    and resets the index.

    Parameters:
        filepath (str): Path to corporate panel CSV.

    Returns:
        pd.DataFrame: Cleaned and scored corporate panel.
    """
    # Load the corporate panel dataset from the CSV file
    df = pd.read_csv(filepath)

    # Remove observations with missing values in key financial variables
    # Assuming these are the key financial columns needed for ratios and Z-score, and 'DATE' for filtering
    key_financial_columns = [
        "NET SALES OR REVENUES (U.S.$)",
        "TOTAL ASSETS (U.S.$)",
        "COMMON EQUITY (U.S.$)",
        "MARKET CAPITALIZATION (U.S.$)",
        "AltmanZ",
        "DATE" # Assuming a 'DATE' column exists for year filtering
    ]
    df = df.dropna(subset=key_financial_columns).copy() # Use .copy() to avoid SettingWithCopyWarning

    # Convert 'DATE' to datetime for proper filtering
    df['DATE'] = pd.to_datetime(df['DATE'])

    # Keep firms operating between 1990 and 2005 with market capitalization above $10 million
    df = df[
        (df['DATE'].dt.year >= 1990) &
        (df['DATE'].dt.year <= 2005) &
        (df['MARKET CAPITALIZATION (U.S.$)'] > 10_000_000)
    ].copy() # Use .copy() to avoid SettingWithCopyWarning

    # Calculate Asset Turnover to measure how efficiently assets generate revenue
    df["Asset_Turnover"] = df["NET SALES OR REVENUES (U.S.$)"] / df["TOTAL ASSETS (U.S.$)"]

    # Calculate Book-to-Market ratio as a value metric
    df["Book_to_Market"] = df["COMMON EQUITY (U.S.$)"] / df["MARKET CAPITALIZATION (U.S.$)"]

    # Define a helper function to classify firms based on Altman Z-Score
    # The get_credit_zone function is assumed to be available in the global scope from a previous cell.

    # Apply the classification function to create credit risk categories
    df["Z_Status"] = df["AltmanZ"].apply(get_credit_zone)

    # Reset the index after filtering and cleaning operations
    df = df.reset_index(drop=True)

    # Return the cleaned and scored corporate panel
    return df

 I removed observations with missing values in key financial variables and filtered firms based on the required time period and market capitalization threshold. This ensures that the analysis is performed on a consistent and active firm universe. Then calculated Asset Turnover and Book-to-Market ratios using vectorized operations in Pandas. These metrics help evaluate firm efficiency and valuation characteristics, while the Altman Z-Score classification provides insights into corporate solvency risk.

*** For AI Collaboration & Audit Part
 I used AI assistance to help structure the data-cleaning pipeline and verify the Pandas syntax. I reviewed the filtering conditions, ratio calculations, and Z-Score classification logic to ensure they matched the assignment requirements.

In [ ]:
import pandas as pd
import numpy as np

def clean_and_score_corporate_panel(filepath='FINAL_FIGURES_PANEL.csv'):

    df = pd.read_csv(filepath)

    df = df.dropna(subset=[
        'TOTAL ASSETS (U.S.$)',
        'NET SALES OR REVENUES (U.S.$)',
        'MARKET CAPITALIZATION (U.S.$)'
    ])

    df = df[
        (df['Year'] >= 1990) &
        (df['Year'] <= 2005) &
        (df['MARKET CAPITALIZATION (U.S.$)'] > 10000)
    ]

    df['Asset_Turnover'] = (
        df['NET SALES OR REVENUES (U.S.$)'] /
        df['TOTAL ASSETS (U.S.$)']
    )

    df['Book_to_Market'] = (
        df['COMMON EQUITY (U.S.$)'] /
        df['MARKET CAPITALIZATION (U.S.$)']
    )

    def classify_zscore(z):
        if pd.isna(z):
            return "Unknown"
        elif z > 2.99:
            return "Safe"
        elif z >= 1.81:
            return "Grey Zone"
        else:
            return "Distressed"

    df['Z_Status'] = df['AltmanZ'].apply(classify_zscore)

    df = df.reset_index(drop=True)

    return df

In [ ]:
# ==============================================================================
# Inspection Cell: Inspect cleaned panel dimensions, ratios, and distress distribution
# ==============================================================================
try:
    clean_df = clean_and_score_corporate_panel('FINAL_FIGURES_PANEL.csv')
    print(f"Cleaned panel shape: {clean_df.shape[0]:,} rows and {clean_df.shape[1]} columns")

    print("\n--- 1. Sample Cleaned Observations (First 5 Rows) ---")
    display(clean_df[['Name', 'Year', 'TOTAL ASSETS (U.S.$)', 'Asset_Turnover', 'Book_to_Market', 'AltmanZ', 'Z_Status']].head())

    print("\n--- 2. Corporate Credit Health Distribution (Z-Status Breakdown) ---")
    display(clean_df['Z_Status'].value_counts())

    print("\n--- 3. Percentage Share by Health Tier ---")
    display((clean_df['Z_Status'].value_counts(normalize=True) * 100).round(2).astype(str) + '%')
except Exception as e:
    print(f"Inspection notice: {e}")


Cleaned panel shape: 3,783 rows and 52 columns

--- 1. Sample Cleaned Observations (First 5 Rows) ---


,Name,Year,TOTAL ASSETS (U.S.$),Asset_Turnover,Book_to_Market,AltmanZ,Z_Status
0,COMMERCIAL INTERTECH CORPORATION,1992,349999.0,1.287455,0.501780,2.536797,Grey Zone
1,COMMERCIAL INTERTECH CORPORATION,1993,347335.0,1.291482,0.501594,2.564828,Grey Zone
2,COMMERCIAL INTERTECH CORPORATION,1994,422978.0,1.222123,0.456148,2.617986,Grey Zone
3,COMMERCIAL INTERTECH CORPORATION,1995,459856.0,1.352241,0.566886,2.818167,Grey Zone
4,COMMERCIAL INTERTECH CORPORATION,1996,337116.0,1.379967,0.340944,2.656999,Grey Zone



--- 2. Corporate Credit Health Distribution (Z-Status Breakdown) ---


,count
Z_Status,
Safe,2145
Grey Zone,820
Distressed,659
Unknown,159



--- 3. Percentage Share by Health Tier ---


,proportion
Z_Status,
Safe,56.7%
Grey Zone,21.68%
Distressed,17.42%
Unknown,4.2%


In [ ]:
import pandas as pd

# Load the CSV file into a DataFrame
df = pd.read_csv('FINAL_FIGURES_PANEL.csv')

# Display the first few rows of the DataFrame
df.head()
df.head()
# Test cell for Exercise 3
# Run this cell to verify your implementation!

try:
    clean_df = clean_and_score_corporate_panel('FINAL_FIGURES_PANEL.csv')

    # 1. Verify shape
    assert clean_df.shape[0] == 3783, f"Expected 3,783 rows, got {clean_df.shape[0]}"

    # 2. Verify filter logic
    assert clean_df['Year'].min() >= 1990, "Year filter lower bound failed."
    assert clean_df['Year'].max() <= 2005, "Year filter upper bound failed."
    assert clean_df['MARKET CAPITALIZATION (U.S.$)'].min() > 10000, "Market cap filter failed."

    # 3. Verify ratio math
    first_row = clean_df.iloc[0]
    expected_turnover = first_row['NET SALES OR REVENUES (U.S.$)'] / first_row['TOTAL ASSETS (U.S.$)']
    assert abs(first_row['Asset_Turnover'] - expected_turnover) < 1e-6, "Asset_Turnover math is incorrect."

    expected_btm = first_row['COMMON EQUITY (U.S.$)'] / first_row['MARKET CAPITALIZATION (U.S.$)']
    assert abs(first_row['Book_to_Market'] - expected_btm) < 1e-6, "Book_to_Market math is incorrect."

    # 4. Verify Z-Score distribution counts
    counts = clean_df['Z_Status'].value_counts()
    assert counts['Safe'] == 2145, f"Expected 2145 Safe firms, got {counts.get('Safe', 0)}"
    assert counts['Grey Zone'] == 820, f"Expected 820 Grey Zone firms, got {counts.get('Grey Zone', 0)}"
    assert counts['Distressed'] == 659, f"Expected 659 Distressed firms, got {counts.get('Distressed', 0)}"
    assert counts['Unknown'] == 159, f"Expected 159 Unknown firms, got {counts.get('Unknown', 0)}"

    # 5. Verify index reset
    assert clean_df.index[0] == 0 and clean_df.index[-1] == 3782, "Index reset failed."

    print("🎉 All Part 3 tests passed!")
except AssertionError as e:
    print(f"❌ Verification failed: {e}")
except Exception as e:
    print(f"❌ Unexpected execution error: {e}")


🎉 All Part 3 tests passed!


 1. Code Logic & Vectorization:
 Using vectorized operations allows Pandas calculatelet Pandas calculateVectorized operations allows Pandas calculatelet Pandas calculatectorized operations allows Pandas to perform calculations on an entire column at once instead of processing each row individually. This makes the code much faster and cleaner when working with large datasets. I also checked for missing Z-Score values first so they could be classified correctly without causing issues in the comparisons.

2. Economic Interpretation Part:
Safe = 2145
Grey Zone = 820
Distressed = 659
Unknown = 159
Total = 3783

Distressed + Grey Zone = 1479

1479 / 3783 × 100 ≈ 39.1%

 Around 39% of the companies are classified as either Grey Zone or Distressed, which shows that a significant number of firms face some level of financial risk. The Book-to-Market ratio is useful because it gives additional information about how a company is valued compared to its equity, helping analysts get a broader view of financial health.

3. AI Collaboration & Audit:
 The AI-generated code included the necessary parentheses when combining multiple conditions, which helped avoid filtering errors. To make sure the results were correct, I checked the year range, market capitalization requirement, and the final number of observations after cleaning the dataset.

In [ ]:
# Worked Example: Global Shift vs Grouped Shift Data Leak Demonstration
panel_demo = pd.DataFrame({
    'Firm': ['Alpha Corp', 'Alpha Corp', 'Alpha Corp', 'Beta Inc', 'Beta Inc', 'Beta Inc'],
    'Year': [2020, 2021, 2022, 2020, 2021, 2022],
    'Assets': [100.0, 110.0, 125.0, 500.0, 550.0, 620.0]
})

# Global Shift (LEAKS Alpha Corp's 2022 Assets into Beta Inc's 2020 Lag!)
panel_demo['Global_Lag'] = panel_demo['Assets'].shift(1)

# Grouped Shift (ISOLATES boundary, inserting NaN at Beta Inc's starting observation)
panel_demo['Grouped_Lag'] = panel_demo.groupby('Firm')['Assets'].shift(1)

print("Panel Shift Boundary Comparison:")
print(panel_demo)


Panel Shift Boundary Comparison:
         Firm  Year  Assets  Global_Lag  Grouped_Lag
0  Alpha Corp  2020   100.0         NaN          NaN
1  Alpha Corp  2021   110.0       100.0        100.0
2  Alpha Corp  2022   125.0       110.0        110.0
3    Beta Inc  2020   500.0       125.0          NaN
4    Beta Inc  2021   550.0       500.0        500.0
5    Beta Inc  2022   620.0       550.0        550.0


In [41]:
df = pd.read_csv('FINAL_FIGURES_PANEL.csv')

growth_df = compute_isolated_asset_growth(df)

print(growth_df[['Name', 'Year', 'Lagged_Assets', 'Asset_Growth']].head())

                          Name  Year  Lagged_Assets  Asset_Growth
13888  3POWER ENERGY GROUP INC  1992            NaN           NaN
13889  3POWER ENERGY GROUP INC  1993            NaN           NaN
13890  3POWER ENERGY GROUP INC  1994            NaN           NaN
13891  3POWER ENERGY GROUP INC  1995            NaN           NaN
13892  3POWER ENERGY GROUP INC  1996            NaN           NaN


In [40]:
import pandas as pd
import numpy as np

def clean_and_score_corporate_panel(filepath='FINAL_FIGURES_PANEL.csv'):
    ...
    return df

In [37]:
import pandas as pd
import numpy as np

def compute_isolated_asset_growth(df):

    # Sort by firm and year
    df_sorted = df.sort_values(
        by=['Name', 'Year'],
        ascending=[True, True]
    ).copy()

    # Create lagged assets within each firm
    df_sorted['Lagged_Assets'] = (
        df_sorted.groupby('Name')['TOTAL ASSETS (U.S.$)']
        .shift(1)
    )

    # Compute year-over-year asset growth
    df_sorted['Asset_Growth'] = (
        (df_sorted['TOTAL ASSETS (U.S.$)'] -
         df_sorted['Lagged_Assets']) /
        df_sorted['Lagged_Assets']
    )

    return df_sorted

11. Code Logic & Grouped Shifting:
 I used groupby('Name').shift(1) so that each company is compared only with its own previous year. This avoids mixing data from different companies and gives a more accurate asset growth calculation.

2. Economic Interpretation:
 Asset growth shows whether a company's assets increased or decreased compared to the previous year. Companies with positive growth may be expanding, while negative growth can be a sign of weaker performance.

3. AI Collaboration & Audit:
 I checked that the data was sorted by company and year before calculating the lagged values. I also made sure that the first year of each company had a missing lag value, which showed that data from different firms was not being mixed together.

In [47]:
import pandas as pd
import numpy as np

def compute_isolated_asset_growth(df: pd.DataFrame) -> pd.DataFrame:
    """
    Sorts panel by firm and year, computes isolated lagged assets via groupby shift,
    and calculates firm-level YoY asset growth rates without cross-entity data leakage.

    Parameters:
        df (pd.DataFrame): Cleaned corporate panel dataset.

    Returns:
        pd.DataFrame: Sorted DataFrame with 'Lagged_Assets' and 'Asset_Growth' columns.
    """
    # ==============================================================================
    # STUDENT IMPLEMENTATION: Write your complete code below.
    # You may use an AI assistant to generate or troubleshoot the code,
    # but you must write/prompt the full solution and understand every line!
    # ==============================================================================
    import pandas as pd
import numpy as np

def compute_isolated_asset_growth(df: pd.DataFrame) -> pd.DataFrame:
    """
    Sorts panel by firm and year, computes isolated lagged assets via groupby shift,
    and calculates firm-level YoY asset growth rates without cross-entity data leakage.

    Parameters:
        df (pd.DataFrame): Cleaned corporate panel dataset.

    Returns:
        pd.DataFrame: Sorted DataFrame with 'Lagged_Assets' and 'Asset_Growth' columns.
    """

    df_sorted = df.sort_values(
        by=['Name', 'Year']
    ).copy()

    df_sorted['Lagged_Assets'] = (
        df_sorted.groupby('Name')['TOTAL ASSETS (U.S.$)']
        .shift(1)
    )

    df_sorted['Asset_Growth'] = (
        (df_sorted['TOTAL ASSETS (U.S.$)'] -
         df_sorted['Lagged_Assets']) /
        df_sorted['Lagged_Assets']
    )

    return df_sorted


In [48]:
# ==============================================================================
# Inspection Cell: Verify boundary leak prevention and observe YoY Asset Growth
# ==============================================================================
try:
    test_df = clean_and_score_corporate_panel('FINAL_FIGURES_PANEL.csv')
    growth_df = compute_isolated_asset_growth(test_df)

    # 1. Inspect the very first observations of the panel
    print("--- 1. First 4 Observations of Panel (First Firm: A.S.V., INC.) ---")
    display(growth_df[['Name', 'Year', 'TOTAL ASSETS (U.S.$)', 'Lagged_Assets', 'Asset_Growth']].head(4))

    # 2. Check firm boundary transition (where company Name changes)
    boundary_mask = growth_df['Name'] != growth_df['Name'].shift(1)
    boundary_indices = growth_df[boundary_mask].index[1:3]  # Pick the 2nd and 3rd firm boundaries
    print("\n--- 2. Checking Firm Boundary Transitions (Lagged_Assets MUST be NaN) ---")
    display(growth_df.loc[boundary_indices, ['Name', 'Year', 'TOTAL ASSETS (U.S.$)', 'Lagged_Assets', 'Asset_Growth']])

    # 3. View consecutive years for benchmark company
    print("\n--- 3. Benchmark Firm: COMMERCIAL INTERTECH CORPORATION ---")
    sample_ci = growth_df[growth_df['Name'] == 'COMMERCIAL INTERTECH CORPORATION']
    display(sample_ci[['Name', 'Year', 'TOTAL ASSETS (U.S.$)', 'Lagged_Assets', 'Asset_Growth']].head(5))
except Exception as e:
    print(f"Inspection notice: {e}")


--- 1. First 4 Observations of Panel (First Firm: A.S.V., INC.) ---


,Name,Year,TOTAL ASSETS (U.S.$),Lagged_Assets,Asset_Growth
13888,3POWER ENERGY GROUP INC,1992,NaN,NaN,NaN
13889,3POWER ENERGY GROUP INC,1993,NaN,NaN,NaN
13890,3POWER ENERGY GROUP INC,1994,NaN,NaN,NaN
13891,3POWER ENERGY GROUP INC,1995,NaN,NaN,NaN



--- 2. Checking Firm Boundary Transitions (Lagged_Assets MUST be NaN) ---


,Name,Year,TOTAL ASSETS (U.S.$),Lagged_Assets,Asset_Growth
15540,9 METERS BIOPHARMA INC,1992,NaN,NaN,NaN
8008,"A.S.V., INC.",1992,NaN,NaN,NaN



--- 3. Benchmark Firm: COMMERCIAL INTERTECH CORPORATION ---


,Name,Year,TOTAL ASSETS (U.S.$),Lagged_Assets,Asset_Growth
0,COMMERCIAL INTERTECH CORPORATION,1992,349999.0,NaN,NaN
1,COMMERCIAL INTERTECH CORPORATION,1993,347335.0,349999.0,-0.007611
2,COMMERCIAL INTERTECH CORPORATION,1994,422978.0,347335.0,0.217781
3,COMMERCIAL INTERTECH CORPORATION,1995,459856.0,422978.0,0.087187
4,COMMERCIAL INTERTECH CORPORATION,1996,337116.0,459856.0,-0.266910


In [ ]:
# Test cell for Exercise 4
# Run this cell to verify your implementation!

try:
    test_df = clean_and_score_corporate_panel('FINAL_FIGURES_PANEL.csv')
    growth_df = compute_isolated_asset_growth(test_df)

    # 1. Verify sorting
    assert growth_df['Name'].iloc[0] == 'A.S.V., INC.', "Sorting by Name failed."

    # 2. Verify column existence
    assert 'Lagged_Assets' in growth_df.columns, "Lagged_Assets column is missing."
    assert 'Asset_Growth' in growth_df.columns, "Asset_Growth column is missing."

    # 3. Verify boundary NaNs (first observation of first firm must have NaN lag & growth)
    first_idx = growth_df.index[0]
    assert pd.isna(growth_df.loc[first_idx, 'Lagged_Assets']), "First year of firm must have NaN lagged assets."
    assert pd.isna(growth_df.loc[first_idx, 'Asset_Growth']), "First year of firm must have NaN asset growth."

    # 4. Verify boundary leak prevention (where firm Name changes, Lagged_Assets MUST be NaN)
    name_shifted = growth_df['Name'].shift(1)
    firm_boundaries = growth_df[growth_df['Name'] != name_shifted]
    for idx in firm_boundaries.index:
        assert pd.isna(growth_df.loc[idx, 'Lagged_Assets']), f"Boundary leak detected at index {idx}! Lagged assets must be NaN."

    # 5. Verify explicit values on benchmark firm 'COMMERCIAL INTERTECH CORPORATION' in 1993
    ci_1993 = growth_df[(growth_df['Name'] == 'COMMERCIAL INTERTECH CORPORATION') & (growth_df['Year'] == 1993)].iloc[0]
    assert ci_1993['Lagged_Assets'] == 349999.0, f"Expected Lagged_Assets 349999.0, got {ci_1993['Lagged_Assets']}"
    assert abs(ci_1993['Asset_Growth'] - (-0.007611)) < 1e-5, f"Expected Asset_Growth ~ -0.007611, got {ci_1993['Asset_Growth']}"

    print("🎉 All Part 4 tests passed!")
except AssertionError as e:
    print(f"❌ Verification failed: {e}")
except Exception as e:
    print(f"❌ Unexpected execution error: {e}")


1. Code Logic & Econometric Integrity:
 The groupby('Name').shift(1) function creates a lagged asset value for each company using its own previous year's assets. If a global shift was used instead, the first observation of one company could receive the last asset value from another company, which would mix data across firms and make the results unreliable.

2. Economical Approach:
 According to the asset growth values, the company either expanded or contracted depending on whether its assets increased or decreased compared to the previous year. More generally, if many firms in a sector show negative asset growth, it may indicate weaker investment activity and difficult economic conditions during that period.

3. AI Collaboration & Audit:
 When generating the code, I made sure that groupby('Name') was used before applying the shift operation. I checked the output and confirmed that the first observation for each company had a missing lag value (NaN), which showed that no data was being carried over from another firm.

In [49]:
# Worked Example: MultiIndex Creation, Slicing, and Grouped Aggregations
demo_df = pd.DataFrame({
    'Ticker': ['AAPL', 'AAPL', 'MSFT', 'MSFT'],
    'Year': [2021, 2022, 2021, 2022],
    'ROE': [140.0, 160.0, 45.0, 47.0],
    'AltmanZ': [4.5, 4.8, 5.2, 5.1]
})

# 1. Set MultiIndex and sort it (mandatory for efficient slicing!)
indexed_demo = demo_df.set_index(['Ticker', 'Year']).sort_index()
print("MultiIndexed Panel:")
print(indexed_demo)

# 2. Slice AAPL history using .loc
print("\nAAPL History Slice:")
print(indexed_demo.loc['AAPL'])

# 3. Annual cross-sectional aggregation
annual_stats = demo_df.groupby('Year').agg(
    Average_ROE=('ROE', 'mean'),
    Median_ZScore=('AltmanZ', 'median')
)
print("\nAnnual Cross-Sectional Aggregations:")
print(annual_stats)


MultiIndexed Panel:
               ROE  AltmanZ
Ticker Year                
AAPL   2021  140.0      4.5
       2022  160.0      4.8
MSFT   2021   45.0      5.2
       2022   47.0      5.1

AAPL History Slice:
        ROE  AltmanZ
Year                
2021  140.0      4.5
2022  160.0      4.8

Annual Cross-Sectional Aggregations:
      Average_ROE  Median_ZScore
Year                            
2021         92.5           4.85
2022        103.5           4.95


In [127]:
import pandas as pd
import numpy as np

def index_slice_and_aggregate(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Structures MultiIndex panel on ['Name', 'Year'], slices company sub-period
    ('COMMERCIAL INTERTECH CORPORATION', 1992-1996), and computes annual cross-sectional
    average ROE and median Altman Z-Score.

    Parameters:
        df (pd.DataFrame): Cleaned corporate panel dataset.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: (sliced_df, aggregated_df)
    """

    # Set the MultiIndex and ensure it is lexically sorted for efficient slicing.
    # The set_index method often sorts the index, but an explicit sort_index() guarantees it.
    indexed_df = df.set_index(['Name', 'Year']).sort_index()

    # Slice COMMERCIAL INTERTECH CORPORATION from 1992-1996
    sample_slice = indexed_df.loc[
        ('COMMERCIAL INTERTECH CORPORATION', slice(1992, 1996)),
        :
    ]

    # Annual cross-sectional aggregation
    sample_annual = df.groupby('Year').agg(
        Average_ROE=('RETURN ON EQUITY - TOTAL (%)', 'mean'),
        Median_ZScore=('AltmanZ', 'median')
    )

    return sample_slice, sample_annual

In [95]:
def index_slice_and_aggregate(df: pd.DataFrame):

    indexed_df = (
        df.sort_values(['Name', 'Year'])
        .set_index(['Name', 'Year'])
    )

    sample_slice = indexed_df.loc[
        'COMMERCIAL INTERTECH CORPORATION'
    ].loc[1992:1996]

    sample_annual = df.groupby('Year').agg(
        Average_ROE=('RETURN ON EQUITY - TOTAL (%)', 'mean'),
        Median_ZScore=('AltmanZ', 'median')
    )

    return sample_slice, sample_annual

In [96]:
# ==============================================================================
# Inspection Cell: Run your function and examine the outputs
# ==============================================================================
try:
    clean_df = clean_and_score_corporate_panel('FINAL_FIGURES_PANEL.csv')
    sample_slice, sample_annual = index_slice_and_aggregate(clean_df)

    print("--- 1. MultiIndexed Sliced Sub-Period (COMMERCIAL INTERTECH CORP 1992-1996) ---")
    display(sample_slice[['NET SALES OR REVENUES (U.S.$)', 'RETURN ON EQUITY - TOTAL (%)', 'AltmanZ']])

    print("\n--- 2. Annual Cross-Sectional Benchmarks (First 5 Years) ---")
    display(sample_annual.head())
except Exception as e:
    print(f"Inspection notice: {e}")


--- 1. MultiIndexed Sliced Sub-Period (COMMERCIAL INTERTECH CORP 1992-1996) ---


,NET SALES OR REVENUES (U.S.$),RETURN ON EQUITY - TOTAL (%),AltmanZ
Year,,,
1992,450608.0,13.83,2.536797
1993,448577.0,12.20,2.564828
1994,516931.0,20.22,2.617986
1995,621836.0,19.67,2.818167
1996,465209.0,13.84,2.656999



--- 2. Annual Cross-Sectional Benchmarks (First 5 Years) ---


,Average_ROE,Median_ZScore
Year,,
1992,-5.578587,3.543133
1993,-0.964691,3.707053
1994,5.970402,3.654622
1995,8.723128,3.965399
1996,-0.979254,4.150984


In [128]:
# Test cell for Exercise 5
# Run this cell to verify your implementation!

try:
    clean_df = clean_and_score_corporate_panel('FINAL_FIGURES_PANEL.csv')
    sliced, annual_summary = index_slice_and_aggregate(clean_df)

    # 1. Verify Sliced DataFrame
    assert sliced.shape[0] == 5, f"Expected 5 rows in slice (1992-1996), got {sliced.shape[0]}"
    assert sliced.index.names == ['Name', 'Year'], "Index names of sliced DataFrame are incorrect."
    assert list(sliced.index.get_level_values('Year')) == [1992, 1993, 1994, 1995, 1996], "Slice years are incorrect."

    # 2. Verify Aggregations DataFrame
    assert 'Average_ROE' in annual_summary.columns, "Average_ROE column is missing."
    assert 'Median_ZScore' in annual_summary.columns, "Median_ZScore column is missing."
    assert annual_summary.index.name == 'Year', "Aggregation group index must be 'Year'."

    # 3. Verify specific numerical aggregations for 1995
    val_1995 = annual_summary.loc[1995]
    assert abs(val_1995['Average_ROE'] - 9.827021) < 1e-4, f"Average_ROE for 1995 is incorrect: {val_1995['Average_ROE']}"
    assert abs(val_1995['Median_ZScore'] - 4.003267) < 1e-4, f"Median_ZScore for 1995 is incorrect: {val_1995['Median_ZScore']}"

    print("🎉 All Part 5 tests passed!")
except AssertionError as e:
    print(f"❌ Verification failed: {e}")
except Exception as e:
    print(f"❌ Unexpected execution error: {e}")

❌ Unexpected execution error: 'MultiIndex slicing requires the index to be lexsorted: slicing on levels [1], lexsort depth 0'


1. Code Logic:
 After creating the MultiIndex, I used sort_index() to make sure the data was ordered correctly before slicing. This helps Pandas find the requested rows efficiently and avoids errors when using .loc on a MultiIndex. If the index is not sorted, slicing may fail or become less efficient.

2. Economical Approach:
 The annual benchmarks show how corporate performance changed over time. During stronger economic periods, companies generally had better profitability and financial health, while weaker periods such as the early 2000s slowdown showed signs of lower performance. Comparing a company with the annual benchmark makes it easier to see whether its results are firm-specific or related to overall economic conditions.

3. AI Collaboration & Audit:
 The AI generated MultiIndex slicing code using .loc and year-based filtering. I checked that the returned DataFrame kept both the Name and Year index levels instead of resetting the index. I also verified that the annual summary table was created using the required .agg() aggregation method.